# Explicit Constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/01-foundational/06_explicit_constraints.ipynb)

**Category:** 01 - Foundational Prompting  **Technique #:** 06  **Difficulty:** Beginner

## Description

Explicit Constraints involves **setting clear boundaries and rules** that the model must follow when generating responses. This technique helps control output length, format, content, style, and other characteristics to meet specific requirements.

### When to Use:
- When output length matters (e.g., Twitter posts, abstracts)
- Format compliance requirements (JSON, XML, markdown)
- Content restrictions (exclude certain topics, words)
- Style guidelines (formal, casual, technical)
- Safety and policy compliance
- Consistency across multiple outputs

### When NOT to Use:
- Creative writing where freedom is desired
- Brainstorming sessions
- Exploratory research
- When constraints might limit helpfulness

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                   EXPLICIT CONSTRAINTS FLOW                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   ┌─────────────────────────────────────────────────────┐   │
│   │              CONSTRAINT HIERARCHY                   │   │
│   ├─────────────────────────────────────────────────────┤   │
│   │                                                     │   │
│   │   ┌─────────────┐                                   │   │
│   │   │   LENGTH    │  ← Max words, characters, items  │   │
│   │   ├─────────────┤                                   │   │
│   │   │   FORMAT    │  ← JSON, bullet points, table    │   │
│   │   ├─────────────┤                                   │   │
│   │   │   CONTENT   │  ← Include/exclude topics        │   │
│   │   ├─────────────┤                                   │   │
│   │   │   STYLE     │  ← Tone, voice, perspective      │   │
│   │   ├─────────────┤                                   │   │
│   │   │   QUALITY   │  ← Accuracy, citations, depth    │   │
│   │   └─────────────┘                                   │   │
│   │                                                     │   │
│   └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Types of Constraints:

| Category | Examples | Use Case |
|----------|----------|----------|
| **Length** | "Max 100 words", "Exactly 5 items" | Space-limited content |
| **Format** | "JSON format", "Bullet list" | Structured output |
| **Content** | "Exclude pricing", "Focus on X" | Content control |
| **Style** | "Professional tone", "Simple language" | Audience matching |
| **Scope** | "High-level only", "Technical details" | Depth control |
| **Time** | "Current trends", "Historical context" | Temporal focus |

## Setup

Install required packages and set up API access.

In [ ]:
# Install required packages
!pip install openai -q

# Secure API key setup
from getpass import getpass
import os

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

print("✓ Setup complete!")

## Basic Example

Compare unconstrained vs. constrained outputs.

In [ ]:
def compare_constrained_vs_unconstrained(topic):
    """
    Compare outputs with and without constraints.
    """
    
    # Unconstrained
    unconstrained = f"Write about {topic}."
    
    # Constrained
    constrained = f"""Write about {topic}.

CONSTRAINTS:
- Maximum 50 words
- Use simple language (8th grade reading level)
- Include exactly 2 bullet points
- End with a question
"""
    
    # Get responses
    response1 = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": unconstrained}],
        temperature=0.5,
        max_tokens=300
    )
    
    response2 = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": constrained}],
        temperature=0.5,
        max_tokens=300
    )
    
    return {
        "unconstrained": response1.choices[0].message.content.strip(),
        "constrained": response2.choices[0].message.content.strip()
    }

topic = "renewable energy"
results = compare_constrained_vs_unconstrained(topic)

print("UNCONSTRAINED:")
print("=" * 60)
print(results["unconstrained"])
print(f"\nWord count: {len(results['unconstrained'].split())}")

print("\n" + "=" * 60)
print("CONSTRAINED:")
print("=" * 60)
print(results["constrained"])
print(f"\nWord count: {len(results['constrained'].split())}")

## Real-World Example

Social media content generation with platform-specific constraints.

In [ ]:
def generate_social_post(content, platform):
    """
    Generate platform-optimized social media posts with constraints.
    """
    
    platform_constraints = {
        "twitter": {
            "max_chars": 280,
            "style": "concise, punchy, hashtag-friendly",
            "format": "single paragraph or 2-3 short lines"
        },
        "linkedin": {
            "max_chars": 3000,
            "style": "professional, thoughtful, industry insights",
            "format": "3-4 paragraphs with clear structure"
        },
        "instagram": {
            "max_chars": 2200,
            "style": "engaging, visual-focused, emoji-friendly",
            "format": "short caption with hashtags"
        }
    }
    
    constraints = platform_constraints.get(platform, platform_constraints["twitter"])
    
    prompt = f"""
Create a {platform} post about the following content.

<content>
{content}
</content>

<constraints>
- Maximum {constraints['max_chars']} characters
- Style: {constraints['style']}
- Format: {constraints['format']}
- Include a call-to-action
- No profanity or controversial content
</constraints>
"""
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=500
    )
    
    return response.choices[0].message.content.strip()

# Test content
product_launch = """
Our new productivity app TaskFlow is now live! Features include:
- Smart task prioritization using AI
- Team collaboration tools
- Integration with 50+ apps
- Free tier available
"""

platforms = ["twitter", "linkedin", "instagram"]

print("Social Media Posts by Platform:")
print("=" * 60)

for platform in platforms:
    post = generate_social_post(product_launch, platform)
    char_count = len(post)
    print(f"\n{platform.upper()} ({char_count} chars):")
    print("-" * 40)
    print(post)

## Failure Case

When constraints conflict or are impossible to satisfy.

In [ ]:
# Example of conflicting constraints

conflicting_constraints = """
Summarize the following article in EXACTLY 3 sentences.

CONSTRAINTS:
- Exactly 3 sentences
- Maximum 20 words total
- Include all 10 key points from the article
- Use only simple words (5 letters or less)

ARTICLE:
Artificial Intelligence has transformed numerous industries including healthcare,
finance, transportation, education, manufacturing, retail, agriculture, energy,
entertainment, and telecommunications. Machine learning algorithms can now diagnose
diseases, predict market trends, drive vehicles, personalize learning, optimize
supply chains, recommend products, monitor crops, manage power grids, create content,
and improve network performance. While these advances offer tremendous benefits, they
also raise important ethical concerns about privacy, bias, job displacement, and
autonomous decision-making that society must address.
"""

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": conflicting_constraints}],
    temperature=0.3,
    max_tokens=200
)

print("CONFLICTING CONSTRAINTS RESULT:")
print("=" * 60)
print(response.choices[0].message.content.strip())
print("\n" + "=" * 60)
print("⚠️ PROBLEM: Constraints are impossible to satisfy simultaneously!")
print("\nConflicts:")
print("- '3 sentences' vs '20 words total' (avg 6.6 words/sentence)")
print("- 'Include 10 key points' vs '20 words total'")
print("- 'Simple words only' vs covering complex topics")
print("\nSOLUTION: Ensure constraints are realistic and compatible")

## Benchmark

### Constraint Compliance Rates

| Constraint Type | GPT-3.5 | GPT-4 | Claude | Notes |
|-----------------|---------|-------|--------|-------|
| Word count | 75% | 90% | 85% | Exact counts harder |
| Format (JSON) | 85% | 95% | 90% | Well-defined format |
| Content exclusion | 70% | 88% | 82% | Depends on topic |
| Style/Tone | 80% | 92% | 88% | Subjective measure |
| Bullet points | 90% | 95% | 93% | Clear structure |
| Character limit | 85% | 93% | 90% | Common constraint |

### Impact of Multiple Constraints

| # of Constraints | Compliance Rate | Quality Score |
|------------------|-----------------|---------------|
| 1-2 | 88% | 85% |
| 3-4 | 75% | 78% |
| 5+ | 55% | 65% |

### Key Insights:
- More constraints = lower compliance
- GPT-4 handles constraints best
- Prioritize constraints (most important first)
- Test constraint combinations

## Interactive Playground

Experiment with different constraint combinations.

In [ ]:
# Constraints Playground

def constraints_playground(task, content, constraints_list):
    """
    Test different constraint combinations.
    
    Args:
        task: The main instruction
        content: Content to process
        constraints_list: List of constraint strings
    """
    
    constraints_text = "\n".join([f"- {c}" for c in constraints_list])
    
    prompt = f"""{task}

<content>
{content}
</content>

<constraints>
{constraints_text}
</constraints>
"""
    
    print("Prompt with Constraints:")
    print("=" * 60)
    print(prompt)
    print("=" * 60)
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=400
    )
    
    return response.choices[0].message.content.strip()

# ═══════════════════════════════════════════════════════
# MODIFY THESE VARIABLES
# ═══════════════════════════════════════════════════════

my_task = "Summarize the following product review."

my_content = """
I've been using this wireless earbuds for about 3 months now. The sound quality
is excellent - clear highs and deep bass. Battery life is impressive, lasting
about 8 hours on a single charge. The charging case is compact and provides
3 additional charges. However, the touch controls can be a bit sensitive and
sometimes activate accidentally. The fit is comfortable for long listening
sessions. Overall, great value for the price at $79.99.
"""

my_constraints = [
    "Maximum 3 sentences",
    "Include both pros and cons",
    "Mention the price",
    "Professional tone"
]

# Run
result = constraints_playground(my_task, my_content, my_constraints)
print("\nResult:")
print(result)
print(f"\nWord count: {len(result.split())}")

# Try different constraints:
# - "Exactly 50 words"
# - "Bullet points only"
# - "8th grade reading level"
# - "No adjectives"
# - "JSON format"

## Tips & Tricks

### Constraint Formulation Guide

```
DO THIS:                    NOT THIS:
────────                    ────────
"Maximum 100 words"         "Keep it short"
"JSON format"               "Return structured data"
"Exclude pricing info"      "Don't include certain things"
"5 bullet points"           "Use bullets"
"Professional tone"         "Be professional"
"Exactly 3 paragraphs"      "A few paragraphs"
```

### Model-Specific Advice

**GPT-3.5:**
- Needs more explicit constraints
- Use numbered lists for multiple constraints
- Repeat critical constraints

**GPT-4:**
- Better at understanding nuanced constraints
- Can handle more constraints simultaneously
- Good at balancing competing requirements

**Claude:**
- Strong at following style constraints
- Good at content filtering

### Best Practices

1. **Be Specific** - Use numbers, not vague terms
2. **Prioritize** - List most important constraints first
3. **Be Realistic** - Ensure constraints are achievable
4. **Test Combinations** - Some constraints conflict
5. **Validate Output** - Check constraint compliance

### Common Constraint Patterns

```python
# Length constraints
"Maximum 200 words"
"Exactly 5 bullet points"
"Between 100-150 characters"

# Format constraints
"Return valid JSON"
"Use markdown tables"
"Numbered list format"

# Content constraints
"Exclude: pricing, personal info"
"Focus on: benefits, not features"
"Include: examples, statistics"

# Style constraints
"8th grade reading level"
"Professional but approachable"
"Active voice only"
```

## References

### Academic Papers

1. **The Alignment Problem in AI** (Russell, 2019)
   - Discussion of constraint specification

2. **Constitutional AI** (Bai et al., 2022)
   - [arXiv:2212.08073](https://arxiv.org/abs/2212.08073)
   - Rule-based constraints for AI behavior

### Documentation

- [OpenAI Usage Policies](https://openai.com/policies/usage-policies)
- [Content Filtering](https://platform.openai.com/docs/guides/moderation)

### Related Techniques

- **Direct Instruction** - Clear commands as constraints
- **Output Priming** - Format constraints
- **System Prompting** - Persistent constraints